# Four-threshold causal live development

This notebook converts the exact four-scale linkage into a causal, stateful read.

The initiating small DCC remains the immutable lineage ID through:

    Stage 1: small → medium
    Stage 2: medium → large
    Stage 3: large → macro

Medium and large DCCs are hand-off events, not replacement seeds. Every live row stops before its stage-resolution bar. Feature timestamps are contemporaneous; propagated/rejected and terminal depth are attached afterward for diagnostic comparison.

The objective is structural intuition, not forecasting or trading.

In [1]:
from enum import StrEnum
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score

from dc_basics.core import dc_live_features, parse_dc_events

DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
SYMBOL = "GBPUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"
SMALL_THETA = 0.002
MEDIUM_THETA = 0.004
LARGE_THETA = 0.006
MACRO_THETA = 0.008
CHECKPOINT_BARS = [0, 5, 10, 20, 40, 80, 160, 320, 640]
CASEBOOK_BARS_BEFORE = 120
CASEBOOK_BARS_AFTER = 120

if not 0.0 < SMALL_THETA < MEDIUM_THETA < LARGE_THETA < MACRO_THETA:
    raise ValueError("Thresholds must satisfy 0 < small < medium < large < macro.")

print(f"Symbol: {SYMBOL}")
print(
    f"Thresholds: {SMALL_THETA:.2%}, {MEDIUM_THETA:.2%}, "
    f"{LARGE_THETA:.2%}, {MACRO_THETA:.2%}"
)

Symbol: GBPUSD.r
Thresholds: 0.20%, 0.40%, 0.60%, 0.80%


In [2]:
class Scale(StrEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"
    MACRO = "macro"


from dc_basics.enums import DcMode


from dc_basics.enums import AttemptOutcome


class TerminalOutcome(StrEnum):
    STOPPED_BELOW_MEDIUM = "stopped_below_medium"
    STOPPED_BELOW_LARGE = "stopped_below_large"
    STOPPED_BELOW_MACRO = "stopped_below_macro"
    REACHED_MACRO = "reached_macro"
    AMBIGUOUS_SMALL_MEDIUM = "ambiguous_small_medium"
    AMBIGUOUS_MEDIUM_LARGE = "ambiguous_medium_large"
    AMBIGUOUS_LARGE_MACRO = "ambiguous_large_macro"
    UNRESOLVED_BELOW_MEDIUM = "unresolved_below_medium"
    UNRESOLVED_BELOW_LARGE = "unresolved_below_large"
    UNRESOLVED_BELOW_MACRO = "unresolved_below_macro"
    UNLINKED_AFTER_MEDIUM = "unlinked_after_medium"
    UNLINKED_AFTER_LARGE = "unlinked_after_large"


class Stage(StrEnum):
    SMALL_TO_MEDIUM = "small_to_medium"
    MEDIUM_TO_LARGE = "medium_to_large"
    LARGE_TO_MACRO = "large_to_macro"


class HierarchyShape(StrEnum):
    BROAD_AGREEMENT = "broad_agreement"
    COUNTER_DEPTH_1 = "counter_depth_1"
    COUNTER_DEPTH_2 = "counter_depth_2"
    COUNTER_DEPTH_3 = "counter_depth_3"
    FRAGMENTED_2_BOUNDARIES = "fragmented_2_boundaries"
    FRAGMENTED_3_BOUNDARIES = "fragmented_3_boundaries"


SCALE_ORDER = (Scale.SMALL, Scale.MEDIUM, Scale.LARGE, Scale.MACRO)
SCALE_SETTINGS = {
    Scale.SMALL: {"prefix": "S", "theta": SMALL_THETA},
    Scale.MEDIUM: {"prefix": "M", "theta": MEDIUM_THETA},
    Scale.LARGE: {"prefix": "L", "theta": LARGE_THETA},
    Scale.MACRO: {"prefix": "X", "theta": MACRO_THETA},
}
STAGE_SETTINGS = {
    Stage.SMALL_TO_MEDIUM: {
        "lower": Scale.SMALL, "upper": Scale.MEDIUM, "attained_depth": 0,
    },
    Stage.MEDIUM_TO_LARGE: {
        "lower": Scale.MEDIUM, "upper": Scale.LARGE, "attained_depth": 1,
    },
    Stage.LARGE_TO_MACRO: {
        "lower": Scale.LARGE, "upper": Scale.MACRO, "attained_depth": 2,
    },
}
MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}

## Stateful information contract

The lineage state is idle until an eligible small DCC leaves broad agreement. It then advances only through exact event-ID hand-offs:

    small seed → medium hand-off → large hand-off → terminal resolution

A rejection at any stage terminates the lineage. Smaller DCCs occurring inside a later stage update nested activity and configuration geometry but never change the original lineage ID.

The live panel excludes every resolution bar, so the current stage outcome is unknown on every feature row.

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))
raw = raw.sort_index()

if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no close column.")
prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all() or prices.le(0.0).any():
    raise ValueError("Close prices must be finite and strictly positive.")

print(f"Bars: {len(prices):,}")
print(f"Range: {prices.index[0]} to {prices.index[-1]}")

Bars: 258,573
Range: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


In [4]:
from dc_basics.events import prepare_event_ledger


event_ledgers = {}
live_frames = {}
mode_frame = pd.DataFrame(index=prices.index)
for scale in SCALE_ORDER:
    settings = SCALE_SETTINGS[scale]
    raw_events = parse_dc_events(prices=prices, theta=settings["theta"])
    ledger = prepare_event_ledger(raw_events, settings["prefix"], scale, settings["theta"])
    live = dc_live_features(prices, theta=settings["theta"])
    event_ledgers[scale] = ledger
    live_frames[scale] = live
    mode_frame[f"{scale.value}_mode"] = live["direction"].map(MODE_MAP)
    mode_frame[f"{scale.value}_upward_dcc"] = live["upward_dcc"].astype(bool)
    mode_frame[f"{scale.value}_downward_dcc"] = live["downward_dcc"].astype(bool)
    event_lookup = ledger.set_index("dcc_idx")["event_id"]
    mode_frame[f"{scale.value}_event_id"] = mode_frame.index.to_series().map(event_lookup)

## Reconstruct and link the three adjacent contests

In [5]:
from dc_basics.hierarchy import build_adjacent_attempts


In [6]:
small_medium_attempts = build_adjacent_attempts(mode_frame, Scale.SMALL, Scale.MEDIUM)
medium_large_attempts = build_adjacent_attempts(mode_frame, Scale.MEDIUM, Scale.LARGE)
large_macro_attempts = build_adjacent_attempts(mode_frame, Scale.LARGE, Scale.MACRO)

previous_modes = mode_frame[
    ["small_mode", "medium_mode", "large_mode", "macro_mode"]
].shift(1)
seed_attempts = small_medium_attempts.copy()
seed_times = pd.DatetimeIndex(seed_attempts["start_time"])
for scale in SCALE_ORDER:
    mode_column = f"{scale.value}_mode"
    seed_attempts[f"prior_{scale.value}_mode"] = previous_modes.loc[
        seed_times, mode_column
    ].to_numpy()
seed_attempts["broad_agreement_before_seed"] = (
    seed_attempts["prior_small_mode"].eq(seed_attempts["prior_medium_mode"])
    & seed_attempts["prior_medium_mode"].eq(seed_attempts["prior_large_mode"])
    & seed_attempts["prior_large_mode"].eq(seed_attempts["prior_macro_mode"])
)
broad_seeds = seed_attempts.loc[seed_attempts["broad_agreement_before_seed"]].copy()

In [7]:
from dc_basics.lineage import build_four_scale_linkages


linkages = build_four_scale_linkages(
    broad_seeds, medium_large_attempts, large_macro_attempts
)

In [8]:
resolved_linkages = linkages.loc[linkages["terminal_depth"].notna()].copy()
stage_one_resolution = resolved_linkages["terminal_position"].where(
    resolved_linkages["terminal_depth"].eq(0), resolved_linkages["medium_position"]
)
stage_two_resolution = resolved_linkages["terminal_position"].where(
    resolved_linkages["terminal_depth"].eq(1), resolved_linkages["large_position"]
)
stage_one_has_live_bar = pd.to_numeric(stage_one_resolution).gt(
    resolved_linkages["seed_position"]
)
stage_two_has_live_bar = (
    resolved_linkages["terminal_depth"].eq(0)
    | pd.to_numeric(stage_two_resolution).gt(
        pd.to_numeric(resolved_linkages["medium_position"])
    )
)
stage_three_has_live_bar = (
    resolved_linkages["terminal_depth"].le(1)
    | pd.to_numeric(resolved_linkages["terminal_position"]).gt(
        pd.to_numeric(resolved_linkages["large_position"])
    )
)
strict_clean_linkages = resolved_linkages.loc[
    resolved_linkages["terminal_label_hindsight_at_seed"]
    & ~resolved_linkages["small_medium_same_bar"]
    & ~resolved_linkages["medium_large_same_bar"]
    & ~resolved_linkages["large_macro_same_bar"]
    & stage_one_has_live_bar
    & stage_two_has_live_bar
    & stage_three_has_live_bar
].copy()
if strict_clean_linkages.empty:
    raise ValueError("No strict-clean four-scale lineages were found.")

cohort_audit = pd.Series({
    "broad_seeds": len(linkages),
    "resolved_lineages": len(resolved_linkages),
    "strict_clean_lineages": len(strict_clean_linkages),
    "stage_one_lineages": len(strict_clean_linkages),
    "stage_two_lineages": int(strict_clean_linkages["terminal_depth"].ge(1).sum()),
    "stage_three_lineages": int(strict_clean_linkages["terminal_depth"].ge(2).sum()),
    "reached_macro": int(strict_clean_linkages["terminal_depth"].eq(3).sum()),
})
cohort_audit

broad_seeds              1006
resolved_lineages        1005
strict_clean_lineages     970
stage_one_lineages        970
stage_two_lineages        439
stage_three_lineages      274
reached_macro             195
dtype: int64

## Causal hierarchy geometry

Running extremes are updated only from prices observed in the active upper-scale mode. Progress is the signed reversal from that causal extreme divided by the upper threshold.

The four-mode configuration contributes additional causal geometry:

- direction boundaries;
- scales matching macro;
- coherent counter depth when a single propagation front exists;
- fragmented versus coherent hierarchy.

In [9]:
from dc_basics.configuration import classify_configuration as _classify_configuration

def classify_configuration(modes: tuple[str, ...]) -> dict[str, object]:
    return _classify_configuration(modes, outer_scale_name="macro", expected_count=4)


mode_columns = [f"{scale.value}_mode" for scale in SCALE_ORDER]
all_modes_known = mode_frame[mode_columns].notna().all(axis=1)
configuration_properties = pd.DataFrame.from_records(
    [
        classify_configuration(tuple(row))
        for row in mode_frame.loc[all_modes_known, mode_columns].itertuples(
            index=False, name=None
        )
    ],
    index=mode_frame.index[all_modes_known],
)
for column in configuration_properties.columns:
    mode_frame.loc[configuration_properties.index, column] = configuration_properties[column]

In [10]:
from dc_basics.live import build_causal_running_extreme


running_extremes = {
    scale: build_causal_running_extreme(prices, live_frames[scale])
    for scale in SCALE_ORDER
}
dcc_cumulative = {}
for scale in (Scale.SMALL, Scale.MEDIUM, Scale.LARGE):
    dcc_flag = (
        mode_frame[f"{scale.value}_upward_dcc"]
        | mode_frame[f"{scale.value}_downward_dcc"]
    ).astype(np.int64)
    dcc_cumulative[scale] = dcc_flag.cumsum().to_numpy(dtype=np.int64)

In [11]:
from dc_basics.audit import audit_live_panel
from dc_basics.lineage import build_wide_stage_table
from dc_basics.live_panel import LivePanelContext, build_live_panel


live_context = LivePanelContext(
    prices=prices,
    mode_frame=mode_frame,
    live_frames=live_frames,
    running_extremes=running_extremes,
    thresholds={scale: SCALE_SETTINGS[scale]["theta"] for scale in SCALE_ORDER},
    dcc_cumulative=dcc_cumulative,
    scale_order=SCALE_ORDER,
    hierarchy_columns=(
        "configuration", "hierarchy_shape", "direction_boundaries",
        "macro_agreement_breadth", "counter_depth", "coherent_frontier",
    ),
)
live_stages = build_wide_stage_table(strict_clean_linkages, SCALE_ORDER)


In [12]:
# Stage rows and causal feature construction are shared in dc_basics.live_panel.


In [13]:
expected_stage_counts = {
    stage.value: int(strict_clean_linkages["terminal_depth"].ge(stage_index).sum())
    for stage_index, stage in enumerate(STAGE_ORDER)
}
live_panel = build_live_panel(live_stages, strict_clean_linkages, live_context, STAGE_SETTINGS)
live_panel_audit = audit_live_panel(
    live_panel,
    expected_stage_counts=expected_stage_counts,
    require_seed_identity=True,
)
live_panel_audit.to_frame()


strict_clean_lineages                 970.00000
active_pre_resolution_rows         166243.00000
stage_one_lineages                    970.00000
stage_two_lineages                    439.00000
stage_three_lineages                  274.00000
maximum_pre_resolution_progress         0.99999
maximum_nested_small_dccs              14.00000
maximum_nested_medium_dccs              5.00000
dtype: float64

## Information-timing audit

Attained depth is causal: it records the highest hand-off already confirmed. Future stage outcome and terminal depth are not causal labels. Stage 1 may already be known to have propagated while later terminal depth remains unresolved.

In [14]:
information_contract = pd.DataFrame([
    ("lineage_id", True, "immutable initiating small event ID"),
    ("stage_start_event_id", True, "small seed, medium hand-off, or large hand-off"),
    ("attained_depth", True, "highest scale confirmed before the feature bar"),
    ("configuration / hierarchy_shape", True, "current four-mode structure"),
    ("direction_boundaries", True, "current fragmentation across adjacent scales"),
    ("macro_agreement_breadth", True, "number of modes agreeing with macro"),
    ("target_progress", True, "current fraction of the active upper threshold"),
    ("lower and upper TMV/T/R", True, "live geometry of the active adjacent clocks"),
    ("nested DCC counts", True, "smaller-scale confirmations after stage start"),
    ("future_stage_outcome", False, "known only on the stage-resolution DCC"),
    ("future_terminal_depth", False, "known only when the complete lineage terminates"),
], columns=["column", "known_on_feature_bar", "economic_meaning"])
information_contract

,column,known_on_feature_bar,economic_meaning
0,lineage_id,True,immutable initiating small event ID
1,stage_start_event_id,True,"small seed, medium hand-off, or large hand-off"
2,attained_depth,True,highest scale confirmed before the feature bar
3,configuration / hierarchy_shape,True,current four-mode structure
4,direction_boundaries,True,current fragmentation across adjacent scales
5,macro_agreement_breadth,True,number of modes agreeing with macro
6,target_progress,True,current fraction of the active upper threshold
7,lower and upper TMV/T/R,True,live geometry of the active adjacent clocks
8,nested DCC counts,True,smaller-scale confirmations after stage start
9,future_stage_outcome,False,known only on the stage-resolution DCC


## Live Audit 1: fixed checkpoints and survival conditioning

At checkpoint k, only stages still unresolved after k bars remain. Later rows describe selected survivors, not the original initiation cohort.

In [15]:
checkpoint_rows = live_panel.loc[live_panel["elapsed_bars"].isin(CHECKPOINT_BARS)].copy()
stage_attempt_counts = live_panel.groupby("stage")["lineage_id"].nunique()

checkpoint_survival = checkpoint_rows.groupby(["stage", "elapsed_bars"], sort=True).agg(
    surviving_attempts=("lineage_id", "nunique"),
    later_propagation_rate=(
        "future_stage_outcome",
        lambda values: values.eq(AttemptOutcome.PROPAGATED.value).mean(),
    ),
    target_progress_median=("target_progress", "median"),
    max_target_progress_median=("max_target_progress", "median"),
    lower_tmv_median=("lower_tmv_live", "median"),
    lower_r_median=("lower_r_live", "median"),
    direction_boundaries_median=("direction_boundaries", "median"),
    macro_agreement_breadth_median=("macro_agreement_breadth", "median"),
    nested_small_dccs_median=("nested_small_dccs", "median"),
    nested_medium_dccs_median=("nested_medium_dccs", "median"),
).reset_index()
checkpoint_survival["survivor_share"] = checkpoint_survival.apply(
    lambda row: row["surviving_attempts"] / stage_attempt_counts[row["stage"]],
    axis=1,
)
checkpoint_survival.set_index(["stage", "elapsed_bars"]).style.format({
    "surviving_attempts": "{:,}",
    "later_propagation_rate": "{:.2%}",
    "target_progress_median": "{:.2%}",
    "max_target_progress_median": "{:.2%}",
    "lower_tmv_median": "{:.3f}",
    "lower_r_median": "{:.3e}",
    "direction_boundaries_median": "{:.1f}",
    "macro_agreement_breadth_median": "{:.1f}",
    "nested_small_dccs_median": "{:.1f}",
    "nested_medium_dccs_median": "{:.1f}",
    "survivor_share": "{:.2%}",
})

In [16]:
checkpoint_outcome_summary = checkpoint_rows.groupby(
    ["stage", "elapsed_bars", "future_stage_outcome", "target_mode"], sort=True
).agg(
    surviving_attempts=("lineage_id", "size"),
    target_progress_median=("target_progress", "median"),
    max_target_progress_median=("max_target_progress", "median"),
    lower_tmv_median=("lower_tmv_live", "median"),
    lower_r_median=("lower_r_live", "median"),
    max_favorable_excursion_median=("max_favorable_excursion_theta", "median"),
    max_adverse_excursion_median=("max_adverse_excursion_theta", "median"),
    direction_boundaries_median=("direction_boundaries", "median"),
    macro_agreement_breadth_median=("macro_agreement_breadth", "median"),
    nested_small_dccs_median=("nested_small_dccs", "median"),
    nested_medium_dccs_median=("nested_medium_dccs", "median"),
)
checkpoint_outcome_summary.style.format({
    "surviving_attempts": "{:,}",
    "target_progress_median": "{:.2%}",
    "max_target_progress_median": "{:.2%}",
    "lower_tmv_median": "{:.3f}",
    "lower_r_median": "{:.3e}",
    "max_favorable_excursion_median": "{:.3f}",
    "max_adverse_excursion_median": "{:.3f}",
    "direction_boundaries_median": "{:.1f}",
    "macro_agreement_breadth_median": "{:.1f}",
    "nested_small_dccs_median": "{:.1f}",
    "nested_medium_dccs_median": "{:.1f}",
})

## Live Audit 2: outcome overlap

Orientation-free AUC measures univariate ordering between the two later stage outcomes among checkpoint survivors. Every feature is causal, but outcome grouping and the AUC are in-sample hindsight diagnostics.

In [17]:
LIVE_FEATURE_COLUMNS = [
    "lower_tmv_live",
    "lower_t_live",
    "lower_r_live",
    "upper_tmv_live",
    "upper_t_live",
    "upper_r_live",
    "target_progress",
    "max_target_progress",
    "max_favorable_excursion_theta",
    "max_adverse_excursion_theta",
    "direction_boundaries",
    "macro_agreement_breadth",
    "nested_small_dccs",
    "nested_medium_dccs",
]

overlap_records = []
for (stage, elapsed_bars), checkpoint in checkpoint_rows.groupby(["stage", "elapsed_bars"], sort=True):
    target = checkpoint["future_stage_outcome"].eq(
        AttemptOutcome.PROPAGATED.value
    ).astype(np.int8)
    for feature in LIVE_FEATURE_COLUMNS:
        values = checkpoint[feature].replace([np.inf, -np.inf], np.nan)
        valid = values.notna()
        feature_values = values.loc[valid]
        feature_target = target.loc[valid]
        if feature_target.nunique() < 2 or feature_values.nunique() < 2:
            raw_auc = np.nan
            stronger_direction = "not_identified"
            orientation_free_auc = np.nan
        else:
            raw_auc = roc_auc_score(feature_target, feature_values)
            stronger_direction = "higher" if raw_auc >= 0.5 else "lower"
            orientation_free_auc = max(raw_auc, 1.0 - raw_auc)
        overlap_records.append({
            "stage": stage,
            "elapsed_bars": elapsed_bars,
            "surviving_attempts": len(checkpoint),
            "propagated_survivors": int(feature_target.sum()),
            "rejected_survivors": int(len(feature_target) - feature_target.sum()),
            "feature": feature,
            "raw_auc_for_higher_value": raw_auc,
            "stronger_value_direction": stronger_direction,
            "orientation_free_auc": orientation_free_auc,
        })

overlap_summary = pd.DataFrame.from_records(overlap_records)
for stage in Stage:
    print(stage.value)
    selected = overlap_summary.loc[
        overlap_summary["stage"].eq(stage.value)
        & overlap_summary["feature"].isin([
            "target_progress", "lower_tmv_live", "lower_r_live",
            "direction_boundaries", "macro_agreement_breadth",
        ])
    ]
    display(selected.pivot(
        index="elapsed_bars",
        columns="feature",
        values="orientation_free_auc",
    ).style.format("{:.3f}"))

small_to_medium


feature,direction_boundaries,lower_r_live,lower_tmv_live,macro_agreement_breadth,target_progress
elapsed_bars,,,,,
0,nan,0.527,0.558,nan,0.573
5,nan,0.571,0.647,nan,0.651
10,nan,0.589,0.663,nan,0.667
20,nan,0.658,0.729,nan,0.731
40,nan,0.661,0.725,nan,0.727
80,nan,0.689,0.733,nan,0.734
160,nan,0.770,0.798,nan,0.788
320,nan,1.000,1.000,nan,0.500


medium_to_large


feature,direction_boundaries,lower_r_live,lower_tmv_live,macro_agreement_breadth,target_progress
elapsed_bars,,,,,
0,nan,0.543,0.516,nan,0.531
5,0.513,0.561,0.635,0.513,0.640
10,0.528,0.566,0.669,0.528,0.668
20,0.597,0.609,0.707,0.597,0.706
40,0.602,0.635,0.715,0.602,0.713
80,0.634,0.702,0.762,0.634,0.760
160,0.580,0.753,0.778,0.580,0.774
320,0.617,0.652,0.712,0.617,0.707
640,nan,0.750,0.750,nan,0.750


large_to_macro


feature,direction_boundaries,lower_r_live,lower_tmv_live,macro_agreement_breadth,target_progress
elapsed_bars,,,,,
0,nan,0.523,0.521,nan,0.530
5,0.521,0.516,0.525,0.521,0.537
10,0.500,0.512,0.594,0.500,0.601
20,0.542,0.522,0.602,0.546,0.608
40,0.599,0.562,0.683,0.616,0.687
80,0.617,0.591,0.692,0.645,0.693
160,0.634,0.583,0.685,0.637,0.687
320,0.575,0.678,0.765,0.683,0.772
640,0.844,1.000,1.000,0.875,1.000


## Live Audit 3: survival-conditioned trajectories

Blue denotes later propagation and orange denotes later rejection. Solid and dashed lines distinguish target direction. Later elapsed bars contain fewer survivors.

In [18]:
from dc_basics.live_plotting import plot_stage_trajectories as _plot_stage_trajectories

def plot_stage_trajectories(panel: pd.DataFrame, stage: Stage, max_elapsed_bars: int) -> go.Figure:
    features = {
        "target_progress": "Progress toward upper DCC",
        "lower_r_live": "Live lower-scale R",
        "direction_boundaries": "Directional boundaries",
        "nested_small_dccs": "Nested small DCCs",
    }
    return _plot_stage_trajectories(
        panel, stage, max_elapsed_bars, symbol=SYMBOL, features=features
    )


plot_stage_trajectories(live_panel, Stage.SMALL_TO_MEDIUM, 200).show()
plot_stage_trajectories(live_panel, Stage.MEDIUM_TO_LARGE, 400).show()
plot_stage_trajectories(live_panel, Stage.LARGE_TO_MACRO, 640).show()

## Live Audit 4: hierarchy-shape occupancy inside each active stage

The current shape is causal. Outcome-conditioned occupancy is hindsight and helps reveal whether later-stage propagation tends to remain coherent or becomes fragmented before resolution.

In [19]:
shape_occupancy = pd.crosstab(
    [
        checkpoint_rows["stage"],
        checkpoint_rows["elapsed_bars"],
        checkpoint_rows["future_stage_outcome"],
    ],
    checkpoint_rows["hierarchy_shape"],
    normalize="index",
)
shape_occupancy.style.format("{:.2%}")

## Terminal-depth live casebook

One recent strict-clean lineage is selected for each terminal depth. Each stage segment contains only pre-resolution feature bars. Later hand-off and terminal markers are hindsight when viewing from the seed.

In [20]:
casebook_lineages = (
    strict_clean_linkages.sort_values("seed_time")
    .groupby("terminal_depth", sort=True, group_keys=False)
    .tail(1)
    .sort_values("terminal_depth")
)
casebook_lineages[[
    "seed_small_event_id", "seed_time", "target_mode",
    "medium_event_id", "medium_time", "large_event_id", "large_time",
    "terminal_time", "terminal_depth",
]]

,seed_small_event_id,seed_time,target_mode,medium_event_id,medium_time,large_event_id,large_time,terminal_time,terminal_depth
1004,S2695,2026-11-06 17:20:00+02:00,up,<NA>,NaT,<NA>,NaT,2026-11-06 21:45:00+02:00,0
998,S2684,2026-10-30 15:45:00+02:00,down,M836,2026-10-30 17:00:00+02:00,<NA>,NaT,2026-11-02 10:35:00+02:00,1
990,S2655,2026-10-13 08:45:00+02:00,up,M829,2026-10-13 20:05:00+02:00,L387,2026-10-14 03:50:00+02:00,2026-10-14 15:00:00+02:00,2
1003,S2694,2026-11-05 21:15:00+02:00,down,M838,2026-11-05 23:05:00+02:00,L392,2026-11-06 08:45:00+02:00,2026-11-06 11:05:00+02:00,3


In [21]:
def plot_live_casebook(close: pd.Series, panel: pd.DataFrame,
                       examples: pd.DataFrame) -> go.Figure:
    figure = make_subplots(
        rows=len(examples),
        cols=1,
        vertical_spacing=min(0.045, 0.16 / len(examples)),
        subplot_titles=[
            (
                f"depth {int(example.terminal_depth)} | target={example.target_mode} | "
                f"seed={example.seed_small_event_id}"
            )
            for example in examples.itertuples(index=False)
        ],
    )
    stage_colors = {
        Stage.SMALL_TO_MEDIUM.value: "#16a34a",
        Stage.MEDIUM_TO_LARGE.value: "#2563eb",
        Stage.LARGE_TO_MACRO.value: "#7c3aed",
    }

    for row_number, example in enumerate(examples.itertuples(index=False), start=1):
        window_start = max(0, int(example.seed_position) - CASEBOOK_BARS_BEFORE)
        window_end = min(
            len(close), int(example.terminal_position) + CASEBOOK_BARS_AFTER + 1
        )
        price_window = close.iloc[window_start:window_end]
        lineage_path = panel.loc[panel["lineage_id"].eq(example.seed_small_event_id)]

        figure.add_trace(go.Scattergl(
            x=price_window.index,
            y=price_window.to_numpy(),
            mode="lines",
            name="Close",
            legendgroup="close",
            showlegend=row_number == 1,
            line={"color": "#94a3b8", "width": 1.0},
            hovertemplate="time=%{x}<br>close=%{y:.5f}<extra></extra>",
        ), row=row_number, col=1)

        for stage, stage_path in lineage_path.groupby("stage", sort=False):
            figure.add_trace(go.Scattergl(
                x=stage_path["time"],
                y=stage_path["price"],
                mode="lines",
                name=stage,
                legendgroup=stage,
                showlegend=row_number == 1,
                line={"color": stage_colors[stage], "width": 3.0},
                customdata=stage_path[[
                    "elapsed_bars", "target_progress", "lower_r_live",
                    "direction_boundaries", "nested_small_dccs",
                ]].to_numpy(),
                hovertemplate=(
                    f"{stage}<br>time=%{{x}}<br>close=%{{y:.5f}}<br>"
                    "elapsed=%{customdata[0]:.0f}<br>progress=%{customdata[1]:.2%}<br>"
                    "live lower R=%{customdata[2]:.3e}<br>"
                    "boundaries=%{customdata[3]:.0f}<br>"
                    "nested small DCCs=%{customdata[4]:.0f}<extra></extra>"
                ),
            ), row=row_number, col=1)

        markers = [
            ("Small seed", example.seed_time, "#111827", "diamond"),
            ("Medium hand-off", example.medium_time, "#2563eb", "circle"),
            ("Large hand-off", example.large_time, "#7c3aed", "square"),
            ("Terminal", example.terminal_time, "#c026d3", "x"),
        ]
        for label, marker_time, color, symbol in markers:
            if pd.isna(marker_time):
                continue
            marker_position = close.index.get_loc(marker_time)
            figure.add_trace(go.Scatter(
                x=[marker_time],
                y=[close.iloc[marker_position]],
                mode="markers",
                name=label,
                legendgroup=label,
                showlegend=row_number == 1,
                marker={"color": color, "size": 9, "symbol": symbol},
                hovertemplate=f"{label}<br>time=%{{x}}<br>price=%{{y:.5f}}<extra></extra>",
            ), row=row_number, col=1)
        figure.update_yaxes(title_text="Close", row=row_number, col=1)

    figure.update_layout(
        title=f"{SYMBOL}: causal four-scale live-development casebook",
        template="plotly_white",
        height=max(850, 350 * len(examples)),
        hovermode="closest",
        legend={"orientation": "h", "y": 1.05, "x": 0.0},
    )
    return figure


plot_live_casebook(prices, live_panel, casebook_lineages).show()

# Four-threshold causal live-development completion gate

Before moving to five thresholds, be able to explain:

1. Why the small event remains the lineage ID through three stages.
2. Why medium and large DCCs change the active stage but not the lineage root.
3. Why each stage trajectory excludes its resolution bar.
4. Why checkpoint-zero progress is already positive.
5. Why attained depth is causal while terminal depth is hindsight.
6. Which TMV/T/R clock is lower and upper in each stage.
7. Why nested small DCCs are expected in later stages.
8. How a Stage-2 or Stage-3 lineage can become fragmented without resolving.
9. What direction-boundary count adds beyond threshold progress.
10. Why macro-agreement breadth does not uniquely identify a coherent propagation front.
11. Why later checkpoints contain survival-selected populations.
12. Why a higher AUC later in the stage is not an initiation-time forecast.
13. Whether live geometry adds information beyond completed initiation features.
14. Why outcome-conditioned trajectories remain diagnostic even though their features are causal.
15. Why none of these results is yet trading alpha.

Once these distinctions are locked in, the next hierarchy milestone is five thresholds and the complete multiscale structural profile.